## Calculating the Units for Multi-family Homes across California

This notebook utilizes parcel data, Zillow data, and building data to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

ERROR 1: PROJ: proj_create_from_database: Open of /Users/sarak/.conda/envs/electrigrid-env/share/proj failed


In [2]:
os.environ['PROJ_LIB'] = '/opt/anaconda3/share/proj'

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

### Parcel data

The analysis relies on parcels to calculate which buildings are assigned to which units. Some buildings intersect with more than one parcel which is kept in both calculations would duplicate the volume of thse homes and cause erroneous unit calculations. Hence a unique parcel number is required to ensure that the calculations accurate.

In [4]:
# load parcels 
parcels = gpd.read_parquet("../../../../../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

In [5]:
# investigate duplicated parcel numbers
#parcels[parcels['PARNO'].duplicated()]


The parcel number should have been unique to each parcel. As shown above there are 525,130 parcel numbers that are not unique. Instead we'll assume that each row is a different parcel and assign a unique id from the index. 

In [6]:
# make an id column in the parcel dataframe to use as the unique id
parcels['parcel_ID'] = parcels.index

In [7]:
parcels.head()

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,parcel_ID
0,48-6298-3-2,Alameda,None,None,None,103.683187,606.327284,"MULTIPOLYGON Z (((-122.12384 37.75571 0.00000,...",0
1,48-6313-23,Alameda,None,None,None,148.045063,1083.135315,"MULTIPOLYGON Z (((-122.12504 37.75429 0.00000,...",1
2,48-6313-87,Alameda,None,None,None,97.520632,557.505860,"MULTIPOLYGON Z (((-122.12635 37.75366 0.00000,...",2
3,48-6330-8-2,Alameda,None,None,None,171.271160,1602.682949,"MULTIPOLYGON Z (((-122.12243 37.75165 0.00000,...",3
4,48-6432-32,Alameda,None,None,None,140.208009,1078.107621,"MULTIPOLYGON Z (((-122.12829 37.76110 0.00000,...",4


### Zillow data

In [8]:
# import Zillow data 
zillow = gpd.read_parquet("../../../../../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("../../../../../../capstone/electrigrid/data/ca_state_boundary.geojson").to_crs(epsg=4326)

# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

In [9]:
print(f"Number of Zillow points", zillow.shape[0])

Number of Zillow points 10012568


We expect to get the same number of homes at the end of unit regression calculation.

### Building Footprint data

Access: https://sat-io.earthengine.app/view/gba

In [10]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

# Unit-regression for multi-family homes and volume calculations for all homes

 Check that subsetting sums up to the total Zillow sum. 


In [11]:
# the analysis only requires the parcel_id and geometry  
parcels = parcels[['parcel_ID', 'geometry']]

## SUBSET ALL THE DATA TYPES
## select only multi-family data from Zillow
zillow_multi_raw = zillow[zillow['type'] == "Multi"]
zillow_multi_raw = zillow_multi_raw[zillow_multi_raw['code'] != "RR106"]

# select the single family home data 
zillow_single_raw = zillow[zillow['type'] == "Single"]

# select the condo data 
zillow_condos_raw = zillow[(zillow['code'] == "RR106") & (zillow['code'] != 'Single' )]

# check to ensure all of the subsets are accurate, these all add up to the total of the zillow 
assert zillow_multi_raw.shape[0] + zillow_single_raw.shape[0] + zillow_condos_raw.shape[0] == zillow.shape[0]

# set a zillow index column so we can track how much zillow data we lose
zillow_multi_raw['zillow_index'] = zillow_multi_raw.index

In [12]:
print(f"There are ",zillow_multi_raw.shape[0], "multi-family zillow.")

There are  603425 multi-family zillow.


In [13]:
zillow_multi_raw.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
8103273,Multi,NaN,NaN,fossil,central,None,4.0,273409.0,living,2548.0,1428589,06025012001,h438,Others,RI103,POINT (-115.48696 32.66592),8103273
8103323,Multi,NaN,NaN,None,None,I,3.0,230000.0,living,3212.0,1428640,06025012001,h438,Others,RI102,POINT (-115.48582 32.66600),8103323
8103319,Multi,NaN,NaN,elec,room,None,2.0,48239.0,living,1682.0,1428636,06025012001,h438,Others,RI101,POINT (-115.48535 32.66603),8103319
8103315,Multi,NaN,NaN,None,None,None,2.0,48770.0,living,1696.0,1428632,06025012001,h438,Others,RI101,POINT (-115.48459 32.66605),8103315
8103242,Multi,NaN,NaN,None,None,None,3.0,74426.0,living,1732.0,1428557,06025012001,h438,Others,RI101,POINT (-115.48818 32.66606),8103242


#### Valid parcel subset

Subset to parcels with multi-family homes.

In [14]:
## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi_raw, predicate="contains")
## output: each parcel where the a multi_zillow point is within it (one-to-many relationship)

# confirm that joining with Zillow decreases the number of parcels
assert len(valid_parcels) < len(parcels)

# drop units as we will add back summed units per parcel after
valid_parcels = valid_parcels.drop(['unit', 'index_right'], axis = 1)

# sum number of units per parcel
units_sum = valid_parcels.sjoin(zillow_multi_raw, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
valid_parcels = valid_parcels.join(units_sum)
valid_parcels.rename(columns={"unit":"total_unit"}, inplace=True)
## for each parcel we should have the total number of units but this means that every time a parcel appears total units appear 

# drop duplicates based on parcel
valid_parcels = valid_parcels.drop_duplicates(subset = 'parcel_ID', keep = 'first')

In [15]:
valid_parcels.head()

,parcel_ID,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
92,92,"MULTIPOLYGON Z (((-122.11883 37.75180 0.00000,...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,123790,2.0
212,212,"MULTIPOLYGON Z (((-122.17728 37.72609 0.00000,...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,103006,4.0
231,231,"MULTIPOLYGON Z (((-122.12955 37.75105 0.00000,...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,123305,2.0
300,300,"MULTIPOLYGON Z (((-122.25473 37.81431 0.00000,...",Multi,1966.0,36.0,None,None,None,8022880.0,living,24529.0,3702,06001403600,574,PGE/SCE,RI104,2970,26.0
318,318,"MULTIPOLYGON Z (((-122.28456 37.81261 0.00000,...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,148788,2.0


In [16]:
print(f"We lose ", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_parcels['zillow_index'].unique()), "homes when assigning to parcels.")

We lose  22481 homes when assigning to parcels.


This could be new developments. The parcel data being from 2014 does not capture all of the current development. These missing homes could be new developments. 
**Assign these homes to their nearest neighbors to calculate their volumes.**

In [17]:
# find the multi-family homes that were not assigned to a parcel
multi_noparcel = zillow_multi_raw[~zillow_multi_raw.zillow_index.isin(valid_parcels['zillow_index'])]

len(multi_noparcel)

22481

As a reminder, `multi_noparcel` contains multi-family Zillow observations that are not within any of the parcels in our data.

In [18]:
# adjust to projected CRS for nearest join 
multi_noparcel = multi_noparcel.to_crs("EPSG:3310")
building = building.to_crs('EPSG:3310')

In [19]:
# rename building index
multi_noparcel = multi_noparcel.rename(columns = {'index__right':'building_index'})

In [20]:
multi_noparcel.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
8102815,Multi,NaN,NaN,fossil,central,None,2.0,79109.0,living,1275.0,1428036,06025012101,h438,Others,RI101,POINT (422809.836 -583611.155),8102815
8103109,Multi,NaN,NaN,fossil,central,O,2.0,151284.0,living,2220.0,1428391,06025012001,h438,Others,RI101,POINT (423702.743 -583556.376),8103109
8114429,Multi,NaN,NaN,fossil,central,None,NaN,323000.0,living,2976.0,1440901,06025012001,h438,Others,RI102,POINT (423625.070 -583434.138),8114429
8103864,Multi,NaN,NaN,fossil,central,O,1.0,248335.0,living,2350.0,1429183,06025012202,h462,Others,RI101,POINT (420938.875 -583028.516),8103864
8102127,Multi,NaN,NaN,fossil,central,None,2.0,275400.0,living,1323.0,1427244,06025012102,h438,Others,RI101,POINT (422750.489 -582665.131),8102127


In [21]:
# Join Zillow points to the nearest building geometry
multi_noparcel = gpd.sjoin_nearest(multi_noparcel,
                                        building,
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_building')

In [22]:
# check length after join to buildings
len(multi_noparcel)

24251

In [23]:
# drop bbox column -- the duplicated function doesn't like dictionaries
multi_noparcel = multi_noparcel.drop('bbox', axis =1)

multi_noparcel.duplicated().sum()

1770

Duplicates created if buildings are equidistant from zillow points.

In [24]:
# drop duplicates
multi_noparcel = multi_noparcel.drop_duplicates()

#### Buildings within multi-family parcels

Assign buildings to parcels which already includes Zillow data. 

In [25]:
# revert building CRS
building = building.to_crs('EPSG:4326')
multi_noparcel = multi_noparcel.to_crs('EPSG:4326')

In [26]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(valid_parcels, predicate="intersects")

# drop the unnecessary column 
valid_buildings = valid_buildings.drop(columns = 'index_right', axis = 1)

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)

In [27]:
valid_buildings.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-121.54641 40.08095, -121.54639 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-121.54630 40.08066, -121.54630 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"{'xmax': -121.87064520706676, 'xmin': -121.870...","POLYGON ((-121.87074 40.43491, -121.87065 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"{'xmax': -121.8706356078727, 'xmin': -121.8707...","POLYGON ((-121.87072 40.43514, -121.87066 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"{'xmax': -121.8708256701087, 'xmin': -121.8709...","POLYGON ((-121.87083 40.43542, -121.87090 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0


In [28]:
print(f"We lose ", len(valid_parcels['parcel_ID'].unique()) - len(valid_buildings['parcel_ID'].unique()), "parcels when assigning buildings")

We lose  9677 parcels when assigning buildings


**Could there potentially be some parcels with no buildings? Or have missing building data?"**

In [29]:
print(f"From raw zillow to buidlings we lose", len(zillow_multi_raw['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()) ,"zillow points")
print(f"From parcels to buildings we lose", len(valid_parcels['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()), "zillow points")

From raw zillow to buidlings we lose 31833 zillow points
From parcels to buildings we lose 9352 zillow points


### Remove parcel duplicates

Buildings can straddle more than one parcel. This causes the volume to be utilized in the regression more than once. To avoid this we assign the building to the parcel where more of the building area is.

In [30]:
# change the crs to a projected crs
building_zillow_parcels = valid_buildings.to_crs("EPSG:6933")
valid_parcels_proj = valid_parcels.to_crs("EPSG:6933").set_index('parcel_ID')

In [31]:
len(building_zillow_parcels)

6913712

In [32]:
len(valid_parcels_proj)

621203

In [33]:
# to fix geometry problems from the crs change
building_zillow_parcels.geometry = building_zillow_parcels.geometry.buffer(0)
valid_parcels_proj.geometry = valid_parcels_proj.geometry.buffer(0)

In [34]:
# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(valid_parcels_proj.loc[building_zillow_parcels['parcel_ID']].geometry.values).area)

In [35]:
len(building_zillow_parcels)

6913712

In [36]:
# keep only the parcel with the largest overlap per building
building_zillow_parcels_merge = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .drop_duplicates(subset=["id", "parcel_ID"], keep="first") # fix -- keep per building, parcel pair
    .drop(columns="intersection_area"))

In [37]:
len(building_zillow_parcels_merge)

6889932

In theory, this value should be greater than the number of multi-family homes in the zillow data, and smaller than the number of `valid_buildings`.

In [38]:
len(building_zillow_parcels_merge) > len(zillow_multi_raw)

True

In [39]:
len(building_zillow_parcels_merge) < len(valid_buildings)

True

In [40]:
building_zillow_parcels = building_zillow_parcels.drop('bbox', axis =1)

building_zillow_parcels.duplicated().sum()

23775

In [41]:
building_zillow_parcels.head()

,source,id,height,var,region,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,intersection_area
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"POLYGON ((-11727561.053 4715034.549, -11727570...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,62.973518
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"POLYGON ((-11727550.366 4715005.976, -11727556...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,44.274674
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"POLYGON ((-11758854.286 4749688.155, -11758858...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,9.786246
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"POLYGON ((-11758852.127 4749710.399, -11758849...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,59.785924
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"POLYGON ((-11758862.640 4749738.070, -11758867...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,74.050334


In [42]:
len(zillow_multi_raw)

603425

In [43]:
print(f"{len(building_zillow_parcels)} vs {len(building_zillow_parcels_merge)}")

6913712 vs 6889932


In [44]:
print(f"Difference that area method makes: {len(building_zillow_parcels) - len(building_zillow_parcels_merge)}")
print(f"Duplicates in original data: {building_zillow_parcels.duplicated().sum()}")

Difference that area method makes: 23780
Duplicates in original data: 23775


So our merge removes 5 buildings that were originally not duplicated...idk how this can be, but better than before!

In [45]:
print(f"We lose",len(valid_buildings['zillow_index'].unique()) - len(building_zillow_parcels['zillow_index'].unique()), "zillow points when trying to remove duplicates." )

We lose 0 zillow points when trying to remove duplicates.


### Calculate volume

**Using valid buildings since we lose so much data when trying to account for the duplicates**

The regression runs on the assumption that volume is linearly correlated with the number of units. First we calculate volume using the area of buildings. Then we run a regression to calculate the number of units where missing in the data.

We currently have three different data subsets:
- multi-family homes (assigned to building through parcels or nearest join)
- single-family homes
- condos

We will calculate volume for each.

#### Multi-family homes

We will concat the `multi_noparcel` and ` valid_buildings` data frames -- `multi_noparcel` will contains NA values in the parcel column.

In [46]:
# have to rename unit column to match "total_unit" in valid_buildings
multi_noparcel = multi_noparcel.rename(columns = {"unit":"total_unit"})

In [47]:
# and reproject data frame to crs with meters as units
building_m = pd.concat([valid_buildings, multi_noparcel]).to_crs("EPSG:6933") 

assert len(valid_buildings) + len(multi_noparcel) == len(building_m)

In [48]:
building_m.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index__right,dist_to_building
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-11727561.053 4715034.549, -11727559...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-11727550.366 4715005.976, -11727550...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"{'xmax': -121.87064520706676, 'xmin': -121.870...","POLYGON ((-11758854.286 4749688.155, -11758845...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"{'xmax': -121.8706356078727, 'xmin': -121.8707...","POLYGON ((-11758852.127 4749710.399, -11758847...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"{'xmax': -121.8708256701087, 'xmin': -121.8709...","POLYGON ((-11758862.640 4749738.070, -11758869...",10631193.0,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0,NaN,NaN


In [49]:
## CALCULATE VOLUME!

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

### Single-family homes

In [50]:
zillow_single_raw.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry
8101290,Single,NaN,NaN,others,central,O,1.0,58405.0,living,728.0,1426294,06025940000,h254,Others,RR103,POINT (-114.63363 32.73575)
8101288,Single,NaN,NaN,None,None,None,1.0,19327.0,living,714.0,1426291,06025940000,h254,Others,RR101,POINT (-114.63382 32.73711)
8101286,Single,NaN,NaN,None,None,O,NaN,28443.0,None,NaN,1426288,06025940000,h254,Others,RR103,POINT (-114.63622 32.73751)
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572)
8103246,Single,NaN,NaN,None,None,O,NaN,114544.0,None,NaN,1428561,06025012001,h438,Others,RR101,POINT (-115.48822 32.66573)


Join Zillow points to buildings for volume calculation.

In [51]:
# adjust to projected CRS for nearest join 
zillow_single = zillow_single_raw.to_crs("EPSG:3310")
building = building.to_crs('EPSG:3310')

# Join single family homes to nearest building (for area and height data)
zillow_single = gpd.sjoin_nearest(zillow_single,
                                        building,
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_building')

In [52]:
print(f"{len(zillow_single_raw)} vs {len(zillow_single)}")

8224373 vs 8392449


Zillow points were added where zillow points are equidistant from buildings. Drop those duplicates.

In [53]:
zillow_single.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index__right,source,id,height,var,region,bbox,dist_to_building
8101290,Single,NaN,NaN,others,central,O,1.0,58405.0,living,728.0,1426294,06025940000,h254,Others,RR103,POINT (503433.743 -572041.017),1729633,ours2,4397,4.540014,0.399243,USA,"{'xmax': -114.63364176102914, 'xmin': -114.633...",7.303619
8101288,Single,NaN,NaN,None,None,None,1.0,19327.0,living,714.0,1426291,06025940000,h254,Others,RR101,POINT (503406.643 -571891.255),1729619,ours2,4313,3.039560,0.324651,USA,"{'xmax': -114.63373600546001, 'xmin': -114.633...",1.495634
8101286,Single,NaN,NaN,None,None,O,NaN,28443.0,None,NaN,1426288,06025940000,h254,Others,RR103,POINT (503179.419 -571859.760),1729590,osm,1219755012,3.524991,0.452001,USA,"{'xmax': -114.6361835, 'xmin': -114.6362982000...",5.337806
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (423675.078 -583964.877),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...",0.000000
8103246,Single,NaN,NaN,None,None,O,NaN,114544.0,None,NaN,1428561,06025012001,h438,Others,RR101,POINT (423696.807 -583962.956),126633,ms,UnitedStates_023013231_18983,3.750597,1.485676,USA,"{'xmax': -115.4880952338817, 'xmin': -115.4883...",0.000000


In [54]:
zillow_single['zillow_index'] = zillow_single.index

# rename building index
zillow_single = zillow_single.rename(columns = {'index__right':'building_index'})

In [55]:
zillow_single = zillow_single.drop("bbox", axis = 1)

zillow_single = zillow_single.sort_values("dist_to_building")
zillow_single = zillow_single.drop_duplicates(subset="zillow_index", keep="first")  # one building per zillow
zillow_single = zillow_single.drop_duplicates(subset="building_index", keep="first")   # one zillow per building

Drops buildings and zillow observations that don't have a match.

In [56]:
print(f"{len(zillow_single_raw)} vs {len(zillow_single)}")

8224373 vs 5506195


In [57]:
len(zillow_single['building_index'].unique())

5506195

We can't calculate the volume without a building, and without dropping the duplicates, we would have multiple buildings matching to a single Zillow point.

In order to calculate area, I join back the building geometries.

In [58]:
print(zillow_single['building_index'].duplicated().sum())

0


In [59]:
# rejoin building geometry
zillow_single = zillow_single.merge(
    building[['geometry']].rename(columns={'geometry': 'building_geometry'}),
    left_on='building_index',
    right_index=True,
    how='left')

# set zillow geometry to building geo
zillow_single = zillow_single.set_geometry('building_geometry')

In [60]:
# deduplicate duplicates from merge -- result of idek what
zillow_single = zillow_single.drop_duplicates(subset="building_index", keep="first")

#### Calculate volume!

In [61]:
# create column from polygon area
zillow_single['area_m2'] = zillow_single.geometry.area

# rename height column to be clear about units
zillow_single.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_single['volume_m3'] = zillow_single['area_m2'] * zillow_single['height_m']

In [62]:
len(zillow_single)

5506195

### Condos (same workflow as single family homes)

In [99]:
# adjust to projected CRS for nearest join 
zillow_condos = zillow_condos_raw.to_crs("EPSG:3310")
building = building.to_crs('EPSG:3310')

# Join single family homes to nearest building (for area and height data)
zillow_condos = gpd.sjoin_nearest(zillow_condos,
                                        building,
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_building')

In [100]:
print(f"{len(zillow_condos_raw)} vs {len(zillow_condos)}")

1184770 vs 1185231


In [102]:
zillow_condos['zillow_index'] = zillow_condos.index

# rename building index
zillow_condos = zillow_condos.rename(columns = {'index__right':'building_index'})

In [103]:
zillow_condos = zillow_condos.drop_duplicates(subset="zillow_index", keep="first")  # one building per zillow

In [125]:
zillow_condos = zillow_condos.drop_duplicates(subset="building_index", keep="first")   # one zillow per building

In [126]:
len(zillow_condos)

149637

In [106]:
zillow_condos.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,building_index,source,id,height,var,region,bbox,dist_to_building,zillow_index
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,8104600
8104552,Multi,NaN,NaN,fossil,central,O,NaN,229500.0,living,1452.0,1429873,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,8104552
8104601,Multi,NaN,NaN,fossil,central,None,NaN,159291.0,living,1276.0,1429922,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,8104601
8104575,Multi,NaN,NaN,fossil,central,I,NaN,204978.0,living,1452.0,1429896,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,8104575
8104622,Multi,NaN,NaN,fossil,central,None,NaN,129170.0,living,1452.0,1429943,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,8104622


In [108]:
len(zillow_condos['geometry'].unique())

301033

In [97]:
zillow_condos = zillow_condos.drop("bbox", axis = 1)

zillow_condos = zillow_condos.sort_values("dist_to_building")
zillow_condos = zillow_condos.drop_duplicates(subset="zillow_index", keep="first")  # one building per zillow
zillow_condos = zillow_condos.drop_duplicates(subset="building_index", keep="first")   # one zillow per building

In [93]:
# rejoin building geometry
zillow_condos = zillow_condos.merge(
    building[['geometry']].rename(columns={'geometry': 'building_geometry'}),
    left_on='building_index',
    right_index=True,
    how='left')

# set zillow geometry to building geo
zillow_condos = zillow_condos.set_geometry('building_geometry')

ValueError: Unknown column building_geometry

In [91]:
# drop duplicates from join
zillow_condos = zillow_condos.drop_duplicates(subset="building_index", keep="first")

In [98]:
len(zillow_condos)

149637

#### Calculate volume!

In [ ]:
# create column from polygon area
zillow_single['area_m2'] = zillow_single.geometry.area

# rename height column to be clear about units
zillow_single.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
zillow_single['volume_m3'] = zillow_single['area_m2'] * zillow_single['height_m']

We are making the assumption that the tax assessor data was correct in its identification of single-family homes and condos, we will assign the `unit` columns in both datasets to be exactly 1 for each observation.

In [ ]:
zillow_single['unit'] = 1
zillow_condos['unit'] = 1

## The rest of the regressio

This part of the workflow only works with the multi-family homes -- the ones that *have* unit data are being used to build the regression, and those with *missing* unit data use that regression to predict the number of units.

In [65]:
# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('parcel_ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'parcel_ID')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_3578671/1524427930.py:43: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_3578671/1524427930.py:44: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [66]:
slope

0.0013897028142430675

In [67]:
intercept

7.445143530376092

#### Utilize the regression to estimate units where they're missing  

In [130]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) | (building_m['total_unit'] == 0)]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'parcel_ID')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

# utilize the values from the regression to calculate units
missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

# calculate each building's share of parcel volume
missing_outlier_units_pred['volume_share'] = (
    missing_outlier_units_pred['volume_m3'] / 
    missing_outlier_units_pred['missing_total_volume_m3'].replace(0, np.nan))

# distribute units proportionally, rounding to nearest int
missing_outlier_units_pred['total_unit'] = round(
    (intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope) 
    * missing_outlier_units_pred['volume_share'])

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual','total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

### Create points from the building polygons

Single family homes and condos are points not polygons. For uniformity and hosting capacity calculation set the points as centroids to the parcels.

#### All multi-family homes

In [131]:
multi_complete['total_unit'].describe()

count    516208.000000
mean         23.375767
std         167.783891
min           0.000000
25%           2.000000
50%           3.000000
75%           6.000000
max       25898.000000
Name: total_unit, dtype: float64

In [138]:
len(multi_complete[multi_complete['total_unit']>2000])

271

In [70]:
# projected CRS
multi_complete = multi_complete.to_crs('EPSG:3310')

# find centroid
multi_complete['centroid'] =  multi_complete.geometry.centroid

In [71]:
#### Single-family homes and condos

zillow_single['centroid'] = zillow_single.geometry.centroid

zillow_condos['centroid'] = zillow_condos.geometry.centroid

In [72]:
# # select only the geometry and parcel id to use for the centroid calculation
# parcels_res = valid_parcels[['parcel_ID', 'geometry']]

# # join the parcel data on ID to get the parcel geometry
# multi_by_parcel = parcels_res.merge(multi_complete, on = 'parcel_ID')

# # set the geometry to the parcel geometry
# multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# # change the crs 
# multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

# # create centroids for each multi residential parcel 
# multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# # drop the geometry of multi_by_parcel so the centroid becomes the geometry
# multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# # rename the centroid to geometry
# multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# # set the centroid to the geometry
# multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# # change the CRS back to the Zillow CRS 
# multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# print(f"We lose ",zillow_multi.shape[0] - len(multi_by_parcel_processed['zillow_index'].unique()), "multi-family zillow points.")

In [73]:
multi_complete.head()

,source,id,height_m,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,index__right,dist_to_building,area_m2,volume_m3,total_volume_m3,volume_share,centroid
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-131839.060 230526.076, -131837.523 ...",480115.0,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0,NaN,NaN,62.973518,283.905634,853.757240,NaN,POINT (-131842.463 230530.570)
26238,osm,759299775,1.686291,0.644541,USA,"{'xmax': -121.43016199999998, 'xmin': -121.430...","POLYGON ((-120940.391 296490.018, -120935.295 ...",10084995.0,Multi,NaN,1.0,None,None,None,448916.0,living,1064.0,9911222,06089012701,302,PGE/SCE,RI101,6245297,2.0,NaN,NaN,38.333495,64.641441,12930.381425,NaN,POINT (-120935.995 296490.155)
30005,ms,UnitedStates_023010010_16,3.319534,0.280181,USA,"{'xmax': -121.69706788384876, 'xmin': -121.697...","POLYGON ((-143127.184 318369.186, -143127.159 ...",10001573.0,Multi,NaN,3.0,None,None,I,96001.0,living,1344.0,9910184,06089012701,357,PGE/SCE,RI101,6244425,1.0,NaN,NaN,283.595485,941.404872,7037.163188,NaN,POINT (-143123.821 318357.812)
31048,ms,UnitedStates_023010010_1720,7.291415,0.815212,USA,"{'xmax': -121.91076675749248, 'xmin': -121.910...","POLYGON ((-161043.960 323729.195, -161035.597 ...",10086613.0,Multi,NaN,3.0,None,None,None,176731.0,living,1476.0,9908514,06089012606,411,PGE/SCE,RI101,6242951,3.0,NaN,NaN,91.268270,665.474852,7736.379409,NaN,POINT (-161037.504 323731.227)
31157,ms,UnitedStates_021232233_5134,6.432827,2.174101,USA,"{'xmax': -121.63049440002435, 'xmin': -121.630...","POLYGON ((-137260.924 332482.055, -137261.604 ...",10000041.0,Multi,NaN,2.0,None,None,None,357000.0,living,3456.0,9908031,06089012701,623,PGE/SCE,RI101,6242607,4.0,NaN,NaN,316.484363,2035.889151,7581.420634,NaN,POINT (-137266.144 332493.328)


In [74]:
zillow_single.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,building_index,source,id,height_m,var,region,dist_to_building,zillow_index,building_geometry,area_m2,volume_m3,centroid
1179440,Single,NaN,NaN,None,None,O,NaN,139549.0,None,NaN,1360886,06023010701,643,PGE/SCE,RR101,POINT (-355099.199 308884.520),256999,ms,UnitedStates_023001111_17786,6.842299,4.605682,USA,0.0,1179440,"POLYGON ((-355086.473 308871.878, -355084.560 ...",162.963729,1115.046556,POINT (-355094.365 308878.952)
2993913,Single,1984.0,3.0,fossil,None,O,NaN,729437.0,living,4886.0,4345454,06053012800,207,PGE/SCE,RR101,POINT (-169974.073 -156577.743),236851,osm,613223076,8.327248,1.963150,USA,0.0,2993913,"POLYGON ((-304145.893 229832.072, -304141.969 ...",124.909635,1040.153457,POINT (-304136.093 229839.915)
2993912,Single,1943.0,4.0,fossil,None,None,NaN,1339714.0,living,5200.0,4345453,06053012800,207,PGE/SCE,RR101,POINT (-170013.128 -156582.294),236843,osm,613223078,9.550425,1.639347,USA,0.0,2993912,"POLYGON ((-304560.582 229790.343, -304559.536 ...",145.981712,1394.187331,POINT (-304550.465 229787.945)
2993918,Single,1980.0,3.0,None,None,O,NaN,371160.0,living,2186.0,4345459,06053012800,207,PGE/SCE,RR101,POINT (-169899.336 -156598.696),236838,osm,613222965,8.714418,2.327320,USA,0.0,2993918,"POLYGON ((-305158.624 230255.692, -305157.037 ...",16.638360,144.993627,POINT (-305155.496 230255.807)
2994000,Single,1952.0,2.0,elec,None,I,NaN,528140.0,living,1520.0,4345541,06053012800,207,PGE/SCE,RR101,POINT (-169748.900 -156552.469),236768,osm,613223011,6.788839,1.013773,USA,0.0,2994000,"POLYGON ((-307633.432 230946.219, -307630.573 ...",197.387641,1340.032885,POINT (-307621.396 230945.669)


In [75]:
zillow_condos.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,building_index,source,id,height_m,var,region,bbox,dist_to_building,building_geometry,area_m2,volume_m3,centroid
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,"POLYGON ((-189593.477 288610.923, -189588.037 ...",39.415792,113.703510,POINT (-189590.624 288614.441)
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,"POLYGON ((424357.805 -582372.606, 424331.180 -...",326.916102,943.061290,POINT (424344.943 -582379.689)
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,"POLYGON ((873003.750 -437348.150, 873008.276 -...",116.914573,337.265760,POINT (873001.140 -437340.981)
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,"POLYGON ((-50464.956 -245016.233, -50466.557 -...",72.010040,207.728773,POINT (-50474.055 -245018.053)
8104600,Multi,NaN,NaN,fossil,central,None,NaN,159515.0,living,1452.0,1429921,06025012003,h279,Others,RR106,POINT (424351.387 -582373.515),124103,ms,UnitedStates_023013231_13671,2.88472,1.064079,USA,"{'xmax': -115.48038605977995, 'xmin': -115.480...",0.0,"POLYGON ((98640.813 -306735.137, 98640.913 -30...",385.813546,1112.963903,POINT (98656.784 -306740.476)


In [139]:
print(f"We have a total of {len(multi_complete) + len(zillow_single) + len(zillow_condos)} observations in our current data. Our original zillow data had {len(zillow)}")

We have a total of 18882198 observations in our current data. Our original zillow data had 10012568


In [140]:
len(zillow_single)

16652252

In [78]:
len(zillow_single_raw)

8224373

In [141]:
len(zillow_condos)

149637

In [80]:
len(zillow_condos_raw)

1184770

In [142]:
len(multi_complete)

2080309

In [82]:
len(zillow_multi_raw)

603425

In [83]:
# takes a really long time 
# multi_by_parcel_processed.to_parquet("multi_zillow_w_units.parquet")

In [110]:
len(zillow_single_raw['geometry'].unique())

7893888

In [115]:
zillow_single_try = zillow_single_raw.sjoin(building, predicate="intersects")

In [114]:
building = building.to_crs(zillow_single_raw.crs)

In [116]:
len(zillow_single_try)

5916622

In [117]:
zillow_single_try.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height,var,region,bbox
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885..."
8103246,Single,NaN,NaN,None,None,O,NaN,114544.0,None,NaN,1428561,06025012001,h438,Others,RR101,POINT (-115.48822 32.66573),126633,ms,UnitedStates_023013231_18983,3.750597,1.485676,USA,"{'xmax': -115.4880952338817, 'xmin': -115.4883..."
8103224,Single,NaN,NaN,elec,room,O,1.0,62187.0,living,1294.0,1428535,06025012101,h438,Others,RR101,POINT (-115.49114 32.66584),126643,ms,UnitedStates_023013231_28389,3.657947,1.622700,USA,"{'xmax': -115.49108985860117, 'xmin': -115.491..."
8103268,Single,NaN,NaN,fossil,room,None,NaN,182272.0,living,1502.0,1428584,06025012001,h438,Others,RR101,POINT (-115.48615 32.66587),126593,ms,UnitedStates_023013231_32434,4.868192,0.718464,USA,"{'xmax': -115.4861119563384, 'xmin': -115.4862..."
8103274,Single,NaN,NaN,elec,room,None,1.0,193520.0,living,884.0,1428590,06025012001,h438,Others,RR101,POINT (-115.48713 32.66592),126582,ms,UnitedStates_023013231_42636,3.861226,1.579459,USA,"{'xmax': -115.48708121028204, 'xmin': -115.487..."


In [120]:
# rejoin building geometry
zillow_single_try = zillow_single_try.merge(
    building[['geometry']].rename(columns={'geometry': 'building_geometry'}),
    left_on='index_right',
    right_index=True,
    how='left')

# set zillow geometry to building geo
zillow_single_try = zillow_single_try.set_geometry('building_geometry')

In [123]:
zillow_single_try.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,index_right,source,id,height,var,region,bbox,building_geometry,area_m2
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...","POLYGON ((-122.21075 40.55556, -122.21074 40.5...",4.691626e-08
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...","POLYGON ((-115.48854 32.66580, -115.48853 32.6...",1.753570e-08
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...","POLYGON ((-111.29062 33.86978, -111.29061 33.8...",1.926886e-08
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...","POLYGON ((-120.88559 35.75266, -120.88554 35.7...",2.135386e-09
8103247,Single,NaN,NaN,None,None,O,NaN,44222.0,None,NaN,1428562,06025012001,h438,Others,RR101,POINT (-115.48846 32.66572),126632,ms,UnitedStates_023013231_19488,3.882336,0.875427,USA,"{'xmax': -115.4883486259241, 'xmin': -115.4885...","POLYGON ((-118.99736 35.30830, -118.99736 35.3...",1.884584e-08


In [122]:
zillow_single_try['area_m2'] = zillow_single_try.geometry.area

/tmp/ipykernel_3578671/3752642535.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zillow_single_try['area_m2'] = zillow_single_try.geometry.area


In [ ]:
zillow_si